In [1]:
import os
import re
import pandas as pd
from nltk.stem import SnowballStemmer

path = '../data/gutenberg/data/'
files = os.listdir(path)[:1000]

stemmer = SnowballStemmer("spanish")

def processor(texto: str) -> str:
    texto =  re.sub(r'[^\w\s]', '', texto).lower()
    tokens = texto.split()
    return ' '.join(stemmer.stem(t) for t in tokens)

raw_docs = []
doc_names = []

for filename in files:
    filepath = os.path.join(path, filename)
    try:
        with open(filepath, encoding='utf-8', errors='ignore') as f:
            raw_docs.append(f.read())
            doc_names.append(filename)
    except Exception as e:
        print(f"Error {filename}: {e}")

df_corpus = pd.DataFrame({'doc': doc_names, 'raw': raw_docs})
df_corpus['processed'] = df_corpus['raw'].apply(processor)

print(f"Documentos cargados: {len(df_corpus)}")
df_corpus.head(3)


KeyboardInterrupt: 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords', quiet=True)
spanish_stopwords = stopwords.words('spanish')

vectorizador = TfidfVectorizer(stop_words=spanish_stopwords)
tfidf_matrix = vectorizador.fit_transform(df_corpus['processed'])

print(f"Matriz TF-IDF: {tfidf_matrix.shape}")  # (docs, vocab)


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

query = "amor y guerra en españa"
query_processed = processor(query)
query_tfidf = vectorizador.transform([query_processed])

scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()
top_indices = np.argsort(scores)[::-1][:5]

print(f"Query: '{query}'\nTop 5 documentos:")
for i in top_indices:
    print(f"  [{scores[i]:.4f}] {df_corpus.iloc[i]['doc']}")
